# spark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RDD_Practical") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark Started Successfully!")

Spark Started Successfully!


26/05/11 12:46:09 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
data = [
    ("Asish", "IT", 50000),
    ("Rahul", "HR", 40000),
    ("Anu", "IT", 60000),
    ("Meera", "Finance", 45000),
    ("John", "HR", 35000),
    ("David", "IT", 70000)
]

In [6]:
rdd = sc.parallelize(data)

rdd.collect()

[('Asish', 'IT', 50000),
 ('Rahul', 'HR', 40000),
 ('Anu', 'IT', 60000),
 ('Meera', 'Finance', 45000),
 ('John', 'HR', 35000),
 ('David', 'IT', 70000)]

In [7]:
print("Partitions:", rdd.getNumPartitions())

Partitions: 4


### Transformation Operations

In [9]:
salary_hike = rdd.map(
    lambda x: (x[0], x[1], x[2]*1.10)
)

salary_hike.collect()

[('Asish', 'IT', 55000.00000000001),
 ('Rahul', 'HR', 44000.0),
 ('Anu', 'IT', 66000.0),
 ('Meera', 'Finance', 49500.00000000001),
 ('John', 'HR', 38500.0),
 ('David', 'IT', 77000.0)]

In [11]:
it_employees = rdd.filter(
    lambda x: x[1] == "IT"
)

it_employees.collect()

[('Asish', 'IT', 50000), ('Anu', 'IT', 60000), ('David', 'IT', 70000)]

In [15]:
departments = rdd.map(lambda x: x[1])

words = departments.flatMap(lambda x: x.split(" "))

words.collect()

['IT', 'HR', 'IT', 'Finance', 'HR', 'IT']

In [16]:
unique_depts = rdd.map(lambda x: x[1]).distinct()

unique_depts.collect()

['HR', 'IT', 'Finance']

In [17]:
sorted_rdd = rdd.sortBy(lambda x: x[2])

sorted_rdd.collect()

[('John', 'HR', 35000),
 ('Rahul', 'HR', 40000),
 ('Meera', 'Finance', 45000),
 ('Asish', 'IT', 50000),
 ('Anu', 'IT', 60000),
 ('David', 'IT', 70000)]

In [18]:
new_data = [
    ("Neha", "Marketing", 55000)
]

rdd2 = sc.parallelize(new_data)
combined = rdd.union(rdd2)
combined.collect()

[('Asish', 'IT', 50000),
 ('Rahul', 'HR', 40000),
 ('Anu', 'IT', 60000),
 ('Meera', 'Finance', 45000),
 ('John', 'HR', 35000),
 ('David', 'IT', 70000),
 ('Neha', 'Marketing', 55000)]

### Actions Operations

In [19]:
#Total employees

rdd.count()

6

In [20]:
rdd.first()

('Asish', 'IT', 50000)

In [21]:
rdd.take(3)

[('Asish', 'IT', 50000), ('Rahul', 'HR', 40000), ('Anu', 'IT', 60000)]

In [23]:
rdd.collect()

[('Asish', 'IT', 50000),
 ('Rahul', 'HR', 40000),
 ('Anu', 'IT', 60000),
 ('Meera', 'Finance', 45000),
 ('John', 'HR', 35000),
 ('David', 'IT', 70000)]

In [24]:
salaries = rdd.map(lambda x: x[2])
total_salary = salaries.reduce(lambda a, b: a+b)
total_salary

300000

In [25]:
pair_rdd = rdd.map(lambda x: (x[1], x[2]))
pair_rdd.collect()

[('IT', 50000),
 ('HR', 40000),
 ('IT', 60000),
 ('Finance', 45000),
 ('HR', 35000),
 ('IT', 70000)]

In [26]:
dept_salary = pair_rdd.reduceByKey(
    lambda a, b: a+b
)
dept_salary.collect()

[('HR', 75000), ('IT', 180000), ('Finance', 45000)]

In [29]:
grouped = pair_rdd.groupByKey()

[(k, list(v)) for k, v in grouped.collect()]

[('HR', [40000, 35000]), ('IT', [50000, 60000, 70000]), ('Finance', [45000])]

In [30]:
count_dept = pair_rdd.countByKey()

count_dept

defaultdict(int, {'IT': 3, 'HR': 2, 'Finance': 1})

### File Processing

In [32]:
text_rdd = sc.textFile("../data/sample.txt")
text_rdd.collect()

['Spark is fast', 'Spark is powerful', 'RDD is important']

In [35]:
word_count = (
    text_rdd
    .flatMap(lambda line: line.split(" "))
    .map(lambda word: (word,1))
    .reduceByKey(lambda a, b: a+b)
)
word_count.collect()

[('fast', 1),
 ('powerful', 1),
 ('important', 1),
 ('Spark', 2),
 ('is', 3),
 ('RDD', 1)]

In [36]:
word_count.saveAsTextFile("../data/output")

In [37]:
spark.stop()